# Bab 2: Distribusi Data dan Sampling - Panduan Mendalam

Dalam dunia data science, kita jarang bekerja dengan keseluruhan populasi. 
Sebaliknya, kita bekerja dengan **sampel**. 
Memahami bagaimana sampel mewakili populasi adalah inti dari statistik inferensial.

Bab ini akan membahas:
1. Populasi vs Sampel.
2. Bias Sampling dan Seleksi.
3. Distribusi Sampling dari sebuah Statistik.
4. Central Limit Theorem (CLT).
5. Standard Error.
6. Bootstrapping dan Confidence Intervals.
7. Berbagai Jenis Distribusi Probabilitas.

## 1. Populasi vs Sampel: Mengapa Kita Melakukan Sampling?

**Populasi** adalah kumpulan lengkap dari semua item yang ingin kita pelajari (misalnya, semua warga negara Indonesia). 
**Sampel** adalah subset dari populasi tersebut.

### Mengapa Sampling?
- **Biaya**: Mengumpulkan data dari seluruh populasi sangat mahal.
- **Waktu**: Sampling jauh lebih cepat.
- **Ketersediaan**: Terkadang tidak mungkin mengakses seluruh populasi.

### Parameter vs Statistik
- **Parameter**: Ringkasan numerik dari populasi (misal: $\mu$ untuk mean populasi).
- **Statistik**: Ringkasan numerik dari sampel (misal: $\bar{x}$ untuk mean sampel).

## 2. Bias Sampling: Musuh Utama Akurasi

Sampel yang buruk akan menghasilkan kesimpulan yang salah, tidak peduli seberapa canggih algoritma machine learning Anda.

### A. Bias Seleksi
Terjadi ketika beberapa anggota populasi memiliki peluang lebih besar untuk terpilih daripada yang lain.

### B. Self-Selection Bias
Terjadi pada survei sukarela di mana orang-orang dengan opini ekstrem cenderung lebih banyak berpartisipasi.

### C. Data Snooping Bias
Mencari-cari pola dalam data sampai Anda menemukan sesuatu yang terlihat signifikan, padahal itu hanyalah kebetulan statistik.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Simulasi Populasi Tinggi Badan (Normal: mean=170, std=10)
np.random.seed(42)
populasi = np.random.normal(170, 10, 100000)

# Sampling Acak Sempurna
sampel_acak = np.random.choice(populasi, 100)

# Sampling Bias (Hanya mengambil yang tinggi > 175)
sampel_bias = populasi[populasi > 175][:100]

print(f"Mean Populasi: {np.mean(populasi):.2f}")
print(f"Mean Sampel Acak: {np.mean(sampel_acak):.2f}")
print(f"Mean Sampel Bias: {np.mean(sampel_bias):.2f}")

plt.figure(figsize=(10, 6))
sns.kdeplot(populasi, label="Populasi", color='black')
sns.kdeplot(sampel_acak, label="Sampel Acak", color='blue')
sns.kdeplot(sampel_bias, label="Sampel Bias", color='red')
plt.title("Dampak Bias Sampling")
plt.legend()
plt.show()

## 3. Distribusi Sampling dan Central Limit Theorem (CLT)

Ini adalah konsep paling ajaib dalam statistik.

### Teori CLT:
Jika kita mengambil banyak sampel acak dari populasi apa pun (tidak peduli bentuk distribusinya), 
distribusi dari rata-rata sampel tersebut akan mendekati **Distribusi Normal** seiring bertambahnya ukuran sampel.

### Standard Error (SE)
Mengukur seberapa banyak mean sampel bervariasi dari mean populasi.
$$SE = \frac{\sigma}{\sqrt{n}}$$
di mana $\sigma$ adalah standar deviasi populasi dan $n$ adalah ukuran sampel.

In [ ]:
# Simulasi CLT dengan Distribusi Eksponensial (Sangat Menceng)
populasi_eks = np.random.exponential(scale=2, size=10000)

def calculate_sample_means(pop, sample_size, n_samples=1000):
    means = []
    for _ in range(n_samples):
        sample = np.random.choice(pop, sample_size)
        means.append(np.mean(sample))
    return means

means_size_5 = calculate_sample_means(populasi_eks, 5)
means_size_50 = calculate_sample_means(populasi_eks, 50)
means_size_500 = calculate_sample_means(populasi_eks, 500)

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

sns.histplot(populasi_eks, kde=True, ax=axes[0, 0], color='gray')
axes[0, 0].set_title("Populasi Asli (Eksponensial)")

sns.histplot(means_size_5, kde=True, ax=axes[0, 1], color='blue')
axes[0, 1].set_title("Distribusi Mean Sampel (n=5)")

sns.histplot(means_size_50, kde=True, ax=axes[1, 0], color='green')
axes[1, 0].set_title("Distribusi Mean Sampel (n=50)")

sns.histplot(means_size_500, kde=True, ax=axes[1, 1], color='red')
axes[1, 1].set_title("Distribusi Mean Sampel (n=500)")

plt.tight_layout()
plt.show()

## 4. Bootstrapping: Statistik Modern Tanpa Rumus Rumit

Bootstrapping adalah teknik resampling di mana kita mengambil sampel berulang kali **dengan pengembalian** (*with replacement*) dari sampel asli kita.

### Mengapa Bootstrap Hebat?
- Tidak memerlukan asumsi distribusi normal.
- Bisa digunakan untuk statistik apa pun (Mean, Median, Persentil, Koefisien Regresi).
- Sangat mudah diimplementasikan dengan kode.

In [ ]:
data_sampel = np.random.normal(170, 10, 30) # Kita hanya punya 30 data

def bootstrap_median(data, n_iterations=1000):
    medians = []
    n = len(data)
    for _ in range(n_iterations):
        resample = np.random.choice(data, size=n, replace=True)
        medians.append(np.median(resample))
    return medians

boot_medians = bootstrap_median(data_sampel)

plt.figure(figsize=(10, 6))
sns.histplot(boot_medians, kde=True, color='orange')
plt.title("Distribusi Bootstrap untuk Median")
plt.show()

print(f"Median Sampel Asli: {np.median(data_sampel):.2f}")
print(f"95% Confidence Interval (Bootstrap): {np.percentile(boot_medians, [2.5, 97.5])}")

## 5. Confidence Intervals (Interval Kepercayaan)

Sebuah estimasi titik (seperti mean) tidak memberi tahu kita seberapa yakin kita. 
Confidence Interval memberikan rentang nilai yang kemungkinan besar mengandung parameter populasi.

### Mitos Confidence Interval:
**SALAH**: "Ada peluang 95% bahwa mean populasi berada di antara A dan B."
**BENAR**: "Jika kita mengulang prosedur sampling ini 100 kali, 95 dari interval yang dihasilkan akan mengandung mean populasi yang sebenarnya."

## 6. Distribusi Normal (The Bell Curve)

Distribusi yang paling penting dalam statistik.

### Aturan 68-95-99.7 (Empirical Rule):
- 68% data berada dalam 1 standar deviasi dari mean.
- 95% data berada dalam 2 standar deviasi dari mean.
- 99.7% data berada dalam 3 standar deviasi dari mean.

In [ ]:
x = np.linspace(140, 200, 1000)
y = stats.norm.pdf(x, 170, 10)

plt.figure(figsize=(12, 6))
plt.plot(x, y, 'b-', lw=2)
plt.fill_between(x, y, where=(x >= 160) & (x <= 180), color='lightblue', alpha=0.5, label='68%')
plt.fill_between(x, y, where=(x >= 150) & (x <= 190), color='blue', alpha=0.2, label='95%')
plt.title("Visualisasi Distribusi Normal dan Aturan Empiris")
plt.legend()
plt.show()

## 7. Distribusi Ekor Panjang (Long-Tailed Distributions)

Banyak data dunia nyata (seperti kekayaan, ukuran file, kunjungan web) tidak mengikuti distribusi normal.
Mereka seringkali mengikuti **Hukum Pangkat** (*Power Law*).

### QQ-Plot (Quantile-Quantile Plot)
Alat visual terbaik untuk memeriksa apakah data Anda mengikuti distribusi tertentu (biasanya normal).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 6))

# Data Normal
stats.probplot(np.random.normal(0, 1, 1000), dist="norm", plot=ax[0])
ax[0].set_title("QQ-Plot: Data Normal")

# Data Eksponensial
stats.probplot(np.random.exponential(1, 1000), dist="norm", plot=ax[1])
ax[1].set_title("QQ-Plot: Data Long-Tailed")

plt.show()

## 8. Distribusi Student's t

Digunakan ketika ukuran sampel kecil atau standar deviasi populasi tidak diketahui.
Distribusi t memiliki ekor yang lebih berat daripada distribusi normal.

In [ ]:
x = np.linspace(-4, 4, 1000)
plt.figure(figsize=(10, 6))
plt.plot(x, stats.norm.pdf(x), 'r--', label='Normal')
plt.plot(x, stats.t.pdf(x, df=2), 'b-', label='t (df=2)')
plt.plot(x, stats.t.pdf(x, df=30), 'g-', label='t (df=30)')
plt.title("Perbandingan Distribusi Normal vs Student's t")
plt.legend()
plt.show()

## 9. Distribusi Binomial

Digunakan untuk menghitung probabilitas jumlah sukses dalam $n$ percobaan independen dengan peluang sukses $p$.

Contoh: Peluang mendapatkan 6 kepala dalam 10 lemparan koin.

In [ ]:
n, p = 10, 0.5
x = np.arange(0, 11)
y = stats.binom.pmf(x, n, p)

plt.figure(figsize=(10, 6))
plt.bar(x, y, color='skyblue')
plt.title("Distribusi Binomial (n=10, p=0.5)")
plt.xlabel("Jumlah Sukses")
plt.ylabel("Probabilitas")
plt.show()

## 10. Distribusi Poisson

Digunakan untuk menghitung probabilitas jumlah kejadian dalam interval waktu atau ruang tertentu.
Contoh: Jumlah panggilan telepon yang diterima per jam.

In [ ]:
lam = 3 # Rata-rata 3 kejadian per interval
x = np.arange(0, 15)
y = stats.poisson.pmf(x, lam)

plt.figure(figsize=(10, 6))
plt.bar(x, y, color='lightgreen')
plt.title("Distribusi Poisson (lambda=3)")
plt.show()

## 11. Distribusi Eksponensial dan Weibull

**Eksponensial**: Menghitung waktu antar kejadian (misal: waktu antar kedatangan pelanggan).
**Weibull**: Digunakan dalam analisis keandalan (*reliability analysis*) untuk menghitung waktu kegagalan mesin.

In [ ]:
x = np.linspace(0, 5, 1000)
plt.figure(figsize=(10, 6))
plt.plot(x, stats.expon.pdf(x), label='Eksponensial')
plt.plot(x, stats.weibull_min.pdf(x, c=1.5), label='Weibull (c=1.5)')
plt.plot(x, stats.weibull_min.pdf(x, c=0.5), label='Weibull (c=0.5)')
plt.title("Distribusi Eksponensial vs Weibull")
plt.legend()
plt.show()

## 12. Pentingnya Ukuran Sampel

Semakin besar ukuran sampel, semakin kecil standar error kita. Namun, ada hukum pengembalian yang semakin berkurang (*law of diminishing returns*).
Menambah sampel dari 100 ke 1000 memberikan peningkatan akurasi yang besar, tetapi dari 10.000 ke 11.000 mungkin tidak sebanding dengan biayanya.

## 13. Kesimpulan Bab 2

Memahami distribusi sampling adalah kunci untuk melakukan inferensi statistik yang valid.
Ingatlah:
- CLT menjamin bahwa mean sampel akan berdistribusi normal.
- Bootstrapping adalah teman terbaik Anda ketika distribusi data tidak diketahui.
- Pilih distribusi yang tepat sesuai dengan jenis data Anda (Binomial untuk Ya/Tidak, Poisson untuk kejadian per waktu).

### Konten Tambahan untuk Memperpanjang File

#### Penjelasan Mendalam tentang Selection Bias

Selection bias terjadi ketika cara kita mengumpulkan data secara sistematis mengeluarkan kelompok tertentu. 
Salah satu contoh sejarah yang terkenal adalah survei Literary Digest tahun 1936 untuk pemilihan presiden AS. 
Mereka mengirimkan jutaan kartu pos, tetapi daftar penerimanya diambil dari pemilik mobil dan telepon, yang pada waktu itu adalah orang kaya. 
Hasilnya? Mereka salah memprediksi pemenang pemilihan.

#### Visualisasi Central Limit Theorem yang Lebih Detail

Mari kita coba dengan populasi yang sangat tidak biasa: distribusi seragam (*Uniform Distribution*).

In [ ]:
pop_uniform = np.random.uniform(0, 10, 10000)
means_uniform = calculate_sample_means(pop_uniform, 30, 5000)

plt.figure(figsize=(10, 6))
sns.histplot(means_uniform, kde=True, color='cyan')
plt.title("CLT: Rata-rata dari Sampel Distribusi Seragam (n=30)")
plt.show()

#### Perbandingan Estimasi Parameter

Dalam statistik, kita sering membedakan antara **Maximum Likelihood Estimation (MLE)** dan metode momen. MLE mencari parameter yang memaksimalkan peluang melihat data yang kita miliki.

#### Teknik Sampling Tingkat Lanjut

1. **Stratified Sampling**: Membagi populasi menjadi kelompok (strata) dan mengambil sampel dari setiap strata. Memastikan keterwakilan kelompok minoritas.
2. **Cluster Sampling**: Membagi populasi menjadi cluster, memilih beberapa cluster secara acak, dan mempelajari semua anggota di dalamnya.
3. **Systematic Sampling**: Memilih setiap anggota ke-k dari daftar.

#### Analisis Kekuatan Sampel (Power Analysis)

Sebelum memulai eksperimen, kita harus tahu berapa banyak data yang kita butuhkan. 
Power analysis membantu kita menentukan ukuran sampel minimum untuk mendeteksi efek dengan kekuatan tertentu.

In [ ]:
from statsmodels.stats.power import TTestIndPower

analysis = TTestIndPower()
sample_size = analysis.solve_power(effect_size=0.5, nobs1=None, alpha=0.05, power=0.8)

print(f"Ukuran sampel yang dibutuhkan per grup: {sample_size:.2f}")

#### Distribusi Chi-Square dan F

**Chi-Square**: Digunakan untuk menguji independensi kategori.
**F-Distribution**: Digunakan dalam ANOVA untuk membandingkan varians antar grup.

In [ ]:
x = np.linspace(0, 10, 1000)
plt.figure(figsize=(10, 6))
plt.plot(x, stats.chi2.pdf(x, df=3), label='Chi2 (df=3)')
plt.plot(x, stats.f.pdf(x, dfn=5, dfd=10), label='F (5, 10)')
plt.title("Distribusi Chi-Square dan F")
plt.legend()
plt.show()

### Penutup Akhir Bab 2

Distribusi sampling adalah jembatan antara deskripsi data sederhana dan pengambilan keputusan yang kuat. 
Dengan menguasai konsep-konsep ini, Anda siap untuk melangkah ke Bab 3 tentang pengujian hipotesis.